In [1]:
from at2v.dloader import MergeSet, TagDataset
from at2v.tokenizer import TagBPETokenizer

In [2]:
print("Training tokenizer..")
tagtok = TagBPETokenizer(vocab_size=10000, min_frequency=2)
data = MergeSet.from_file("../data/output/merged_tags.json")
extended_data = data.extend_with_synthetic(perm_limit=5, sub_array_count=5)
print(f"Total examples {len(extended_data)}")
tagtok.train(extended_data, "../ayo.json")
print("Done!")

Training tokenizer..
Total examples 131592
Tokenizer saved to ../ayo.json
Done!


In [3]:
print(tagtok.encode("1girl glasses handsup"))
print(tagtok.encode("glasses"))
print(tagtok.encode("dsa𓂀"))  # not in vocab
print(tagtok.encode_ids("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"))
print(len(tagtok.encode_ids(" ".join(data.real_examples[5000]))))

['1', 'girl', 'Ġglasses', 'Ġhand', 'su', 'p']
['glasses']
['d', 'sa', '[UNK]', 'ĵ', 'Ĥ', 'Ģ']
[33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 8129, 53, 54, 55, 56, 57, 58, 474, 64, 1010, 67, 68, 177, 71, 72, 73, 74, 406, 77, 78, 79, 321, 82, 83, 84, 3859, 87]
17


In [4]:
from torch.utils.data import DataLoader

max_len_cut = 64
batch_size = 64

dataset = TagDataset(data.tags, tokenizer=tagtok, max_len_cut=max_len_cut)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [5]:
import torch.nn as nn
import torch
import torch.nn.functional as F

In [6]:
buff = 100
d_model = 128

emb = nn.Embedding(
    num_embeddings=tagtok.vocab_size + buff,
    embedding_dim=d_model,
)
emb(torch.tensor([tagtok.vocab_size])).shape, emb(dataset[1]).shape

(torch.Size([1, 128]), torch.Size([64, 128]))

In [7]:
n_layers = 3
n_heads = 4

transformer = nn.TransformerEncoder(
    encoder_layer=nn.TransformerEncoderLayer(
        d_model=d_model, nhead=n_heads, batch_first=True
    ),
    num_layers=n_layers,
    enable_nested_tensor=False
)

In [8]:
output_emb = 10
linproj = nn.Linear(2*d_model, output_emb)

In [9]:
def forward(x: torch.Tensor):
    ix = x                    # (B, I)
    print(ix.shape)
    x = emb(ix)               # (B, D, D)
    x = transformer(x)        # (B, D, D)
    x = torch.cat([
        x.mean(dim=1),        # (B, D) context
        x.max(dim=1).values   # (B, D) highlights
    ], dim=-1)                # (B, 2D)
    ox = linproj(x)           # (B, O)
    return ox

batch = torch.stack([
    dataset[0],
    dataset[8]
])

forward(batch).shape

torch.Size([2, 64])


torch.Size([2, 10])

In [10]:
# TODO
# https://arxiv.org/abs/2407.00143
import torch.nn.functional as F

def augment_tags(x, drop_prob=0.15, shuffle=True):
    mask = torch.bernoulli(torch.full(x.shape, 1 - drop_prob)).to(x.device)
    x_augmented = x * mask.long()
    if shuffle:
        for i in range(x_augmented.size(0)):
            perm = torch.randperm(x_augmented.size(1))
            x_augmented[i] = x_augmented[i][perm]
    return x_augmented

def compute_loss(fwd, batch_data, temperature=0.07):
    view1 = augment_tags(batch_data) 
    view2 = augment_tags(batch_data)
    emb1 = fwd(view1) # (B, O)
    emb2 = fwd(view2) # (B, O)
    emb1 = F.normalize(emb1, p=2, dim=1)
    emb2 = F.normalize(emb2, p=2, dim=1)
    logits = (emb1 @ emb2.T) / temperature # (B, B) where diagonal is self-similarity
    loss = F.cross_entropy(
        logits,                    
        torch.arange(emb1.size(0)) # target is the diagonal (item 0 in view1 == item 0 in view2)
    )

    return loss

compute_loss(forward, batch)

torch.Size([2, 64])
torch.Size([2, 64])


tensor(0.5383, grad_fn=<NllLossBackward0>)